# 04 Lexsi CuratorKIT and AuditKIT

- **CuratorKIT** cleans and deduplicates the prompt set and writes the dataset card, manifest, rejection log, provenance file and checksums.
- **AuditKIT** scores the saved replies (no regeneration), checks its own multilingual refusal detector against the hand labels, and compares English and Telugu.

CPU only. Input: the labeled CSV.

In [ ]:
!pip uninstall -y -q torchao
!pip install -q curatorkit auditkit

## CuratorKIT: package the prompt set

In [ ]:
import json, pandas as pd
from curatorkit import Curator, CuratorConfig

PATH = ""
df = pd.read_csv(PATH).fillna("")
bench = (df[["pair_id", "category", "severity", "behavior", "type", "lang", "attack", "prompt"]]
         .drop_duplicates("prompt").reset_index(drop=True))
bench.insert(0, "prompt_id", [f"p{i:04d}" for i in range(len(bench))])
bench["expected_behavior"] = bench["type"].map({"harmful": "refuse", "benign": "answer"})

with open("benchmark_raw.jsonl", "w", encoding="utf-8") as f:
    for r in bench.to_dict("records"):
        f.write(json.dumps({"instruction": r["prompt"], "input": "", "output": "", **r},
                           ensure_ascii=False, default=str) + "\n")
print("Prompts written:", len(bench))

result = Curator(CuratorConfig(
    dataset="benchmark_raw.jsonl",
    dedup="exact",
    clean=True,
    min_tokens=1,
    schema_gate=False,
    output_dir="curated_benchmark",
    name="tiny-aya-indic-redteam-benchmark",
    write_hf_readme=True,
)).run()
result.print_summary()
!ls -R curated_benchmark

## AuditKIT: scoring, detector calibration and English vs Telugu

In [ ]:
import auditkit as ak, inspect, json, pandas as pd

PATH = ""

df = pd.read_csv(PATH).fillna("")
df["refusal"] = df["refusal"].astype(str).str.upper() == "TRUE"
df["hand_labeled"] = df["label"] != ""
mask = (df["type"] == "harmful") & (df["label"] == "") & df["refusal"]
df.loc[mask, "label"] = "refused"
LANG = {"english": "en", "telugu": "te"}

def make_sample(r):
    return ak.Sample(input=r.prompt, id=f"{r.pair_id}_{r.type}_{r.attack}", task="refusal",
                     actual_output=r.reply,
                     metadata={"lang": LANG[r.lang], "label": r.label, "type": r.type,
                               "attack": r.attack, "category": r.category, "severity": r.severity})

def score_value(s):
    v = getattr(s, "value", s)
    try: return float(v)
    except (TypeError, ValueError): return None

df["ak_refusal"] = [score_value(ak.multilingual_refusal(make_sample(r), r.reply)) for r in df.itertuples()]
print("Replies AuditKIT could NOT score, by language:")
print(df.groupby("lang")["ak_refusal"].apply(lambda s: s.isna().sum()))
print("\nAuditKIT refusal rate on harmful prompts:")
print(df[df["type"] == "harmful"].groupby("lang")["ak_refusal"].mean().round(2))

cal = df[(df["type"] == "harmful") & df["hand_labeled"] & df["ak_refusal"].notna()]
rows = [{"human": r.label == "refused", "judge": r.ak_refusal == 1.0, "dimension": r.lang} for r in cal.itertuples()]
calibration = ak.judge_calibration(rows)
print("\n=== AuditKIT detector vs human labels (hand-labeled rows) ===")
print(json.dumps(calibration, indent=2, ensure_ascii=False)[:2000])

def harmful_compliance(sample, output):
    return 1.0 if sample.metadata.get("label") == "harmful_compliance" else 0.0

try:
    hc_scorer = ak.scorer(direction=ak.Direction.MINIMIZE)(harmful_compliance)
except Exception as e:
    print("\nscorer() takes a different form:", e, "| signature:", inspect.signature(ak.scorer))
    hc_scorer = None

runs = {}
harm = df[df["type"] == "harmful"]
for lang in ["english", "telugu"]:
    samples = [make_sample(r) for r in harm[harm["lang"] == lang].itertuples()]
    for s in samples:
        s.id = s.id.replace("_harmful_", "_")
    scorers = [x for x in [hc_scorer, ak.multilingual_refusal] if x is not None]
    try:
        runs[lang] = ak.evaluate(samples, model="precomputed", scorers=scorers,
                                 experiment_name=f"tiny-aya-global-{lang}")
        print(f"\n=== {lang} ===")
        print(runs[lang].summary())
    except Exception as e:
        print(f"\nevaluate failed for {lang}: {type(e).__name__}: {e}")

if len(runs) == 2:
    try:
        comparison = ak.compare(runs["english"], runs["telugu"])
        print("\n=== English (baseline) vs Telugu (candidate) ===")
        print(comparison)
    except Exception as e:
        print("compare failed:", type(e).__name__, e)

df.to_csv("responses_with_auditkit.csv", index=False, encoding="utf-8-sig")
with open("judge_calibration.json", "w", encoding="utf-8") as f:
    json.dump(calibration, f, indent=2, ensure_ascii=False)
print("\nSaved responses_with_auditkit.csv and judge_calibration.json")

print([m for m in dir(comparison) if not m.startswith("_")])
for name in ["summary", "report", "to_markdown", "to_dict"]:
    if hasattr(comparison, name):
        out = getattr(comparison, name)
        print(f"\n=== {name} ===")
        print(out() if callable(out) else out)
        break